# model 1 — traditional ML prototype
*messy working notebook, lets goooo*

**Sean McManus — ML/DNN Lead**  
goal: load processed data → EDA → train models → compare → SHAP  
this is NOT the final train.py, just figuring out what works

## 1. imports

In [ ]:
import sys
from pathlib import Path

# notebook lives in notebooks/ so project root is one level up
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# sklearn stuff
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# xgboost — might not be installed yet, handle gracefully
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('xgboost not installed — run: pip install xgboost')
    # can still run LR and RF without it

pd.set_option('display.max_columns', 60)
print('imports done')

## 2. load data
loading the processed file that cliff's pipeline already cleaned up  
raw file is 101k rows — processed should be around 94-99k after filtering deceased patients

In [ ]:
# load the processed data — this is what the pipeline spits out
proc_path = PROJECT_ROOT / 'data' / 'processed' / 'patient_encounters_2023_processed.csv'
df = pd.read_csv(proc_path, low_memory=False)

print(f'processed shape: {df.shape}')
# checking shape first — if this looks wrong something went bad in the pipeline
df.head(3)

## 3. quick EDA
just want to see whats in the data before i throw it at a model

In [ ]:
# check target distribution — is it imbalanced?
# imbalanced means way more 0s than 1s which screws with the model
print('readmission counts:')
print(df['readmission_binary'].value_counts())
print(f"\nreadmission rate: {df['readmission_binary'].mean()*100:.1f}%")
# if this is way off from 50/50 we need class_weight='balanced' in our models

In [ ]:
# plot the class split — want to see it visually
df['readmission_binary'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('readmission class distribution')
plt.xticks([0, 1], ['no readmit (0)', 'readmit (1)'], rotation=0)
plt.ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
# check for any remaining missing values
# pipeline should have handled these but double checking
missing = df.isnull().sum()
if missing.sum() == 0:
    print('no missing values — pipeline did its job')
else:
    print('still some missing values:')
    print(missing[missing > 0])
    # if there are missing values, the imputer in the model pipeline will catch them
    # but good to know whats still messy

In [ ]:
# look at numeric feature distributions
# want to make sure nothing looks insane (like a column thats all zeros)
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['readmission_binary', 'encounter_id', 'patient_nbr']]

# just the first 12 so the chart isnt a disaster
df[num_cols[:12]].hist(figsize=(16, 10), bins=30, color='steelblue', edgecolor='white')
plt.suptitle('numeric feature distributions (first 12)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# what features correlate most with readmission?
# this is a sneak peek at what the model will probably care about
corr_with_target = df[num_cols + ['readmission_binary']].corr()['readmission_binary'].drop('readmission_binary')

corr_with_target.abs().sort_values(ascending=False).head(15).plot(
    kind='barh', color='steelblue'
)
plt.title('top 15 features correlated with readmission')
plt.xlabel('|correlation|')
plt.tight_layout()
plt.show()

# correlation doesnt mean causation but gives us a good starting point

## 4. prep features for modeling
need to:
- drop id columns (encounter_id, patient_nbr) — these are just row identifiers, not features
- separate X (features) and y (target)
- handle any leftover object columns
- split into train/val

In [ ]:
# columns to drop — IDs and the raw target (keeping readmission_binary)
# dont want the model cheating by seeing encounter_id or patient_nbr
DROP_COLS = ['encounter_id', 'patient_nbr', 'Unnamed: 0', 'readmitted']
TARGET = 'readmission_binary'

# separate target before dropping anything
y = df[TARGET].astype(int)

# drop IDs and target from features
X = df.drop(columns=[c for c in DROP_COLS + [TARGET] if c in df.columns])

print(f'X shape: {X.shape}')
print(f'y distribution: {y.value_counts().to_dict()}')

In [ ]:
# check for any leftover object (string) columns
# cant feed strings into sklearn models
obj_cols = X.select_dtypes(include='object').columns.tolist()
if obj_cols:
    print(f'found object columns, one-hot encoding: {obj_cols}')
    X = pd.get_dummies(X, columns=obj_cols, drop_first=True)
else:
    print('no object columns left — good to go')

# force everything numeric just in case
X = X.apply(pd.to_numeric, errors='coerce')

print(f'final feature count: {X.shape[1]}')
feature_names = X.columns.tolist()

In [ ]:
# train/val split — MUST use stratify=y because the classes are imbalanced
# without stratify, the val set might have a completely different ratio than train
# which would make our AUC scores meaningless
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,   # 42 so results are reproducible
    stratify=y
)

print(f'train: {len(X_train):,} rows')
print(f'val:   {len(X_val):,} rows')
print(f'train readmission rate: {y_train.mean()*100:.1f}%')
print(f'val readmission rate:   {y_val.mean()*100:.1f}%')
# these two rates should be very close — thats what stratify does

## 5. train models
trying three algorithms — logistic regression first as a baseline, then RF and XGBoost  
using sklearn Pipeline so imputer + model are bundled together  
that way the imputer only fits on training data and transforms val the same way

In [ ]:
# --- logistic regression (baseline) ---
# simplest model, good for checking if the data is reasonable
# class_weight='balanced' handles the class imbalance automatically
lr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # fill any NaN with median
    ('scaler', StandardScaler()),                   # LR needs scaled features
    ('clf', LogisticRegression(
        max_iter=1000,           # needs more iterations for this many features
        class_weight='balanced', # tells the model to weight the minority class more
        random_state=42
    ))
])

print('training logistic regression...')
lr.fit(X_train, y_train)
lr_auc = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
print(f'logistic regression AUC: {lr_auc:.4f}')

In [ ]:
# --- random forest ---
# tree-based, doesnt need scaling, handles non-linear patterns well
# n_jobs=-1 means use all CPU cores — faster training
rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=200,        # 200 trees — more is usually better up to a point
        max_depth=10,            # limit depth to avoid overfitting
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

print('training random forest...')
rf.fit(X_train, y_train)
rf_auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
print(f'random forest AUC: {rf_auc:.4f}')

In [ ]:
# --- xgboost ---
# usually the best performer on tabular data
# scale_pos_weight handles class imbalance differently than class_weight
# scale_pos_weight = (# negative samples) / (# positive samples)
# roughly 3 here since readmission rate is ~47% ... actually let me check
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f'scale_pos_weight should be: {scale:.2f}')

if XGBOOST_AVAILABLE:
    xgb = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,      # slow and steady — lower LR usually = better
            scale_pos_weight=scale,  # using the calculated value above
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1
        ))
    ])

    print('training xgboost...')
    xgb.fit(X_train, y_train)
    xgb_auc = roc_auc_score(y_val, xgb.predict_proba(X_val)[:, 1])
    print(f'xgboost AUC: {xgb_auc:.4f}')
else:
    print('xgboost not available, skipping')

## 6. compare models

In [ ]:
# build a results dict — add xgb only if it ran
results = {
    'Logistic Regression': lr_auc,
    'Random Forest': rf_auc
}
if XGBOOST_AVAILABLE:
    results['XGBoost'] = xgb_auc

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['AUC-ROC'])
results_df = results_df.sort_values('AUC-ROC', ascending=False)

print(results_df)
print(f"\nbenchmark (0.70): {'✅ met' if results_df['AUC-ROC'].max() >= 0.70 else '❌ not met'}")
print(f"stretch   (0.80): {'✅ met' if results_df['AUC-ROC'].max() >= 0.80 else 'not yet'}")

# bar chart comparison
results_df['AUC-ROC'].plot(kind='barh', color='steelblue', xlim=(0.5, 1.0))
plt.axvline(x=0.70, color='red', linestyle='--', label='benchmark (0.70)')
plt.axvline(x=0.80, color='green', linestyle='--', label='stretch (0.80)')
plt.title('model comparison — AUC-ROC')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# full classification report on the best model
# AUC is nice but want to see precision/recall breakdown too
best_name = results_df.index[0]
model_lookup = {'Logistic Regression': lr, 'Random Forest': rf}
if XGBOOST_AVAILABLE:
    model_lookup['XGBoost'] = xgb

best_model = model_lookup[best_name]

print(f'best model: {best_name}')
print(classification_report(y_val, best_model.predict(X_val), target_names=['No Readmit', 'Readmit']))
# watch out for low recall on Readmit — means model is missing real readmissions
# in clinical settings, a false negative (missing a readmission) is worse than a false positive

In [ ]:
# confusion matrix — visual version of above
# top left = correctly predicted no readmit
# bottom right = correctly predicted readmit
# bottom left = missed readmissions (false negatives — bad!)
# top right = false alarms (false positives)
cm = confusion_matrix(y_val, best_model.predict(X_val))
disp = ConfusionMatrixDisplay(cm, display_labels=['No Readmit', 'Readmit'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False)
ax.set_title(f'confusion matrix — {best_name}')
plt.tight_layout()
plt.show()

## 7. SHAP feature importance
SHAP tells us WHY the model made each prediction — not just what it predicted  
each dot in the plot = one patient  
position left/right = pushed prediction toward or away from readmission  
color red = high feature value, blue = low feature value  
this is what makes the model explainable to clinicians

In [ ]:
# install shap if you haven't yet: pip install shap
try:
    import shap

    # pull the actual classifier out of the pipeline
    # cant run SHAP on the whole pipeline, needs just the model part
    clf = best_model.named_steps['clf']

    # run X_train through the pipeline steps BEFORE the classifier
    # so the data is in the same format the classifier saw during training
    X_transformed = X_train.copy()
    for step_name, step in best_model.steps[:-1]:
        X_transformed = step.transform(X_transformed)

    # TreeExplainer works for XGBoost and Random Forest
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_transformed)

    # Random Forest returns a list (one per class) — grab class 1 (readmit)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # dot plot — shows direction and magnitude
    print('SHAP summary plot (dots):')
    shap.summary_plot(
        shap_values, X_transformed,
        feature_names=feature_names,
        max_display=20
    )

except ImportError:
    print('shap not installed — run: pip install shap')
except Exception as e:
    print(f'shap error: {e}')

In [ ]:
# bar plot version — cleaner, shows overall importance without direction
# easier to read at a glance for the presentation
try:
    shap.summary_plot(
        shap_values, X_transformed,
        feature_names=feature_names,
        max_display=20,
        plot_type='bar'
    )
except:
    # shap_values might not exist if the cell above failed
    print('run the cell above first')

## 8. notes / next steps
- TODO: tune hyperparameters — grid search on XGBoost n_estimators / max_depth
- TODO: look at which features SHAP flags — talk to the team about clinical relevance
- TODO: move best model config into train.py once happy with it
- TODO: build Model 2 (DNN) and compare — does deep learning actually beat XGBoost here?
- NOTE: if recall on Readmit class is low, try lowering the prediction threshold from 0.5 → 0.3